In [11]:
# INSTALL DEPENDENCIES
# pip install nibabel
# pip install matplotlib

import math
import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt
import math

from dash import Dash, dcc, html, Input, Output
from dash.exceptions import PreventUpdate
import plotly.express as px
import os
from functools import lru_cache

# =========================
# 1) LOAD IMAGE (backend)
# =========================
# BASE_DIR = "/Users/cassidyschuman/Downloads/Classes/Senior Spring/Clinical Preceptorship/example_ouptut/"
BASE_DIR = "/Users/forrestlin/Downloads/example_output"

NETWORK_COLORS = [
    np.array([0.9, 0.1, 0.1]),   # red
    np.array([0.1, 0.6, 0.1]),   # green
    np.array([0.1, 0.3, 0.9]),   # blue
    np.array([0.9, 0.6, 0.1]),   # orange
    np.array([0.6, 0.1, 0.8]),   # purple
    np.array([0.1, 0.8, 0.8]),   # cyan
    np.array([0.8, 0.2, 0.5]),   # pink
    np.array([0.5, 0.5, 0.1]),   # olive
    np.array([0.3, 0.7, 0.5]),   # teal
    np.array([0.7, 0.3, 0.3]),   # brick
    np.array([0.3, 0.3, 0.7]),   # indigo
    np.array([0.7, 0.7, 0.3])    # yellow-green
]

# Discover patient folders automatically
patients = sorted([
    d for d in os.listdir(BASE_DIR)
    if os.path.isdir(os.path.join(BASE_DIR, d))
])

def overlay_network_maps(volumes, axis, idx, threshold):
    """
    Create an RGB overlay of multiple brain network maps.
    """

    rgb = None

    for i, vol in enumerate(volumes):

        sl = get_slice(vol, axis, idx)

        sl_norm = normalize_to_unit(sl)

        sl_thresh = np.where(sl_norm >= threshold, sl_norm, 0)

        color = NETWORK_COLORS[i]

        colored = np.dstack([
            sl_thresh * color[0],
            sl_thresh * color[1],
            sl_thresh * color[2]
        ])

        if rgb is None:
            rgb = colored
        else:
            rgb += colored

    rgb = np.clip(rgb, 0, 1)

    return rgb

@lru_cache(maxsize=64)
def load_patient_volume(patient_id, filename):

    path = os.path.join(BASE_DIR, patient_id, "xprx", "display", filename)

    img = nib.load(path)
    vol = img.get_fdata().astype(np.float32)
    if vol.ndim == 4:
        vol = vol[..., 0]

    vol = np.nan_to_num(vol)
    return vol

axis_labels = {0: "Sagittal (x)", 1: "Coronal (y)", 2: "Axial (z)"}

def get_display_niftis(patient_id):
    """
    Returns dict:
    {
        "seedcorr": "full_filename.nii.gz",
        "reho": "full_filename.nii.gz",
        ...
    }
    """
    display_path = os.path.join(BASE_DIR, patient_id, "xprx", "display")

    nii_files = [
        f for f in os.listdir(display_path)
        if f.endswith(".nii") or f.endswith(".nii.gz")
    ]

    file_dict = {}

    prefix = f"sub-{patient_id}_ses-1_task-rest_acq-multiecho_"

    for f in nii_files:
        name = f.replace(".nii.gz", "").replace(".nii", "")

        if name.startswith(prefix):
            suffix = name.replace(prefix, "")
            file_dict[suffix] = f

    return file_dict

def get_slice(vol3d, axis, idx):
    """Extract a 2D slice from 3D volume."""
    if axis == 0:
        sl = vol3d[idx, :, :]
    elif axis == 1:
        sl = vol3d[:, idx, :]
    else:
        sl = vol3d[:, :, idx]
    return np.rot90(sl)


def clip_percentile(sl, p_low=1, p_high=99):
    """Clip intensities for display (robust contrast)."""
    lo, hi = np.percentile(sl, (p_low, p_high))
    if hi <= lo:
        return np.zeros_like(sl, dtype=np.float32)
    return np.clip(sl, lo, hi).astype(np.float32)


def make_mosaic(vol3d, axis=2, start=0, step=5, n_slices=36, pad=2):
    """Tile multiple slices into one mosaic image."""
    max_idx = vol3d.shape[axis] - 1
    start = int(np.clip(start, 0, max_idx))
    step = max(1, int(step))
    n_slices = max(1, int(n_slices))

    indices = list(range(start, max_idx + 1, step))[:n_slices]
    if len(indices) == 0:
        indices = [start]

    slices = []
    for i in indices:
        sl = get_slice(vol3d, axis, i)
        sl = clip_percentile(sl, 1, 99)
        slices.append(sl)

    # grid close to square
    cols = int(math.ceil(math.sqrt(len(slices))))
    rows = int(math.ceil(len(slices) / cols))

    h, w = slices[0].shape
    mosaic_h = rows * h + (rows - 1) * pad
    mosaic_w = cols * w + (cols - 1) * pad
    mosaic = np.zeros((mosaic_h, mosaic_w), dtype=np.float32)

    for k, sl in enumerate(slices):
        r = k // cols
        c = k % cols
        y0 = r * (h + pad)
        x0 = c * (w + pad)
        mosaic[y0:y0 + h, x0:x0 + w] = sl

    return mosaic, indices

def normalize_to_unit(vol):
    vmin = np.min(vol)
    vmax = np.max(vol)
    if vmax == vmin:
        return np.zeros_like(vol)
    return (vol - vmin) / (vmax - vmin)

def build_network_legend(network_names):
    items = []
    for i,name in enumerate(network_names):
        color = NETWORK_COLORS[i]
        rgb = tuple((color*255).astype(int))
        items.append(
            html.Div(
                name,
                style={
                    "color": f"rgb{rgb}",
                    "fontWeight": "bold"
                }
            )
        )
    return items


In [12]:
# ---- 2) Build Dash UI (frontend) ----
app = Dash(__name__)  # keep it strict; no suppress_callback_exceptions needed

app.layout = html.Div(
    style={"maxWidth": "1100px", "margin": "20px auto", "fontFamily": "system-ui"},
    children=[
        html.H2("MRI Viewer (Single Slice + Mosaic)"),
        
        html.Div(
            style={"marginBottom": "20px", "width": "300px"},
            children=[
                html.Label("Patient"),
                dcc.Dropdown(
                    id="patient_dropdown",
                    options=[{"label": p, "value": p} for p in patients],
                    value=patients[0],
                    clearable=False,
                    ),
                ],
            ),
            
        html.Div(
            style={"marginBottom": "20px", "width": "300px"},
            children=[
                html.Label("Map Type"),
                dcc.Dropdown(
                    id="nifti_dropdown",
                    multi=True,
                ),
            ],
        ),

        # Controls row 1
        html.Div(
            style={"display": "flex", "gap": "18px", "alignItems": "center", "flexWrap": "wrap"},
            children=[
                html.Div(
                    style={"width": "240px"},
                    children=[
                        html.Label("Mode"),
                        dcc.RadioItems(
                            id="mode",
                            options=[
                                {"label": "Single slice", "value": "single"},
                                {"label": "Mosaic", "value": "mosaic"},
                            ],
                            value="single",
                            inline=True,
                        ),
                    ],
                ),
                html.Div(
                    style={"width": "260px"},
                    children=[
                        html.Label("Axis"),
                        dcc.Dropdown(
                            id="axis",
                            options=[{"label": axis_labels[i], "value": i} for i in [0, 1, 2]],
                            value=2,
                            clearable=False,
                        ),
                    ],
                ),
            ],
        ),

        html.Hr(),
        html.Div(id="network_legend"),
        # Controls row 2 (always present -> avoids "nonexistent object" errors)
        html.Div(
            style={"display": "flex", "gap": "18px", "alignItems": "center", "flexWrap": "wrap"},
            children=[
                html.Div(style={"flex": "1", "minWidth": "520px"}, children=[
                    html.Label("Slice index (base)"),
                    dcc.Slider(
                        id="idx",
                        min=0,
                        max=100,      # temporary placeholder
                        step=1,
                        value=50,     # temporary placeholder
                    )
                ]),
                html.Div(style={"flex": "1", "minWidth": "320px"}, children=[
                    html.Label("Mosaic: number of slices"),
                    dcc.Slider(
                        id="mosaic_n",
                        min=4, max=100, step=4, value=36,
                        marks={4: "4", 16: "16", 36: "36", 64: "64", 100: "100"},
                        tooltip={"placement": "bottom", "always_visible": False},
                        updatemode="drag",
                    ),
                ]),
            ],
        ),

        html.Div(
            style={"flex": "1", "minWidth": "320px"},
            children=[
                html.Label("Magnitude Threshold"),
                dcc.Slider(
                    id="threshold",
                    min=0,
                    max=1,
                    step=0.01,
                    value=0.0,
                    marks={
                        0: "0",
                        0.25: "0.25",
                        0.5: "0.5",
                        0.75: "0.75",
                        1: "1",
                    },
                    tooltip={"placement": "bottom", "always_visible": False},
                    updatemode="drag",
                ),
            ],
        ),

        html.Div(id="note", style={"marginTop": "10px", "color": "#555"}),

        dcc.Graph(id="view", style={"height": "850px"}),
    ],
)

In [ ]:
# ---- 3) Callback 1: keep slider max/value in sync with axis ----
@app.callback(
    Output("nifti_dropdown", "options"),
    Output("nifti_dropdown", "value"),
    Input("patient_dropdown", "value"),
)
def update_nifti_dropdown(patient_id):

    file_dict = get_display_niftis(patient_id)

    if not file_dict:
        return [], None

    options = [
        {"label": k, "value": v}
        for k, v in sorted(file_dict.items())
    ]

    first_value = [options[0]["value"]]

    return options, first_value


@app.callback(
    Output("idx", "max"),
    Output("idx", "value"),
    Output("note", "children"),
    Input("patient_dropdown", "value"),
    Input("nifti_dropdown", "value"),
    Input("axis", "value"),
    Input("mode", "value"),
    Input("mosaic_n", "value"),
)

def sync_axis(patient_id, filename, axis, mode, mosaic_n):
    if patient_id is None or filename is None:
        raise PreventUpdate

    if axis is None:
        raise PreventUpdate
    
    vol = load_patient_volume(patient_id, filename[0])
    if axis is None:
        raise PreventUpdate

    axis = int(axis)

    max_idx = vol.shape[axis] - 1
    mid = max_idx // 2

    note = (
        f"Volume shape: {vol.shape} | "
        f"axis: {axis_labels[axis]} (n={max_idx+1}) | "
        f"mode: {mode}"
    )
    if mode == "mosaic":
        note += f" | mosaic_n: {mosaic_n}"

    return max_idx, mid, note


@app.callback(
    Output("view", "figure"),
    Input("patient_dropdown", "value"),
    Input("nifti_dropdown", "value"),
    Input("threshold", "value"),
    Input("mode", "value"),
    Input("axis", "value"),
    Input("idx", "value"),
    Input("mosaic_n", "value"),
)

def update_view(patient_id, filename, threshold, mode, axis, idx, mosaic_n):
    if patient_id is None or filename is None:
        raise PreventUpdate

    if mode is None or axis is None or idx is None or mosaic_n is None:
        raise PreventUpdate
    
    volumes = [load_patient_volume(patient_id, f) for f in filename]
    # Normalize volume to 0–1
    # vol_norm = normalize_to_unit(vol)
    # Apply magnitude threshold
    # vol_thresh = np.where(vol_norm >= threshold, vol_norm, 0)

    if mode is None or axis is None or idx is None or mosaic_n is None:
        raise PreventUpdate

    axis = int(axis)
    idx = int(idx)
    mosaic_n = int(mosaic_n)

    max_idx = volumes[0].shape[axis] - 1

    if mode == "single":

        snapped = max(0, min(idx, max_idx))

        rgb = overlay_network_maps(volumes, axis, snapped, threshold)

        fig = px.imshow(rgb)

        fig.update_layout(
            margin=dict(l=10, r=10, t=40, b=10),
            title=f"{axis_labels[axis]} slice {snapped} | 12-network overlay",
        )

        return fig

    # Mosaic mode
    start = int(np.clip(idx, 0, max_idx))

    mosaic, indices = make_mosaic(
        volumes[0],
        axis=axis,
        start=start,
        step=1,
        n_slices=mosaic_n,
        pad=2,
    )

    # Normalize mosaic image itself
    mosaic_norm = normalize_to_unit(mosaic)

    # Apply threshold relative to mosaic
    mosaic_thresh = np.where(mosaic_norm >= threshold, mosaic_norm, 0)

    fig = px.imshow(mosaic_thresh, color_continuous_scale="gray")
    fig.update_layout(
        margin=dict(l=10, r=10, t=40, b=10),
        coloraxis_showscale=False,
        title=f"Mosaic | {axis_labels[axis]} | threshold={threshold:.2f}",
    )
    return fig


if __name__ == "__main__":
    app.run(debug=True)

[2026-03-16 09:57:46,969] ERROR in app: Exception on /_dash-update-component [POST]
Traceback (most recent call last):
  File "/Users/forrestlin/Library/Python/3.11/lib/python/site-packages/flask/app.py", line 917, in full_dispatch_request
    rv = self.dispatch_request()
         ^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/forrestlin/Library/Python/3.11/lib/python/site-packages/flask/app.py", line 902, in dispatch_request
    return self.ensure_sync(self.view_functions[rule.endpoint])(**view_args)  # type: ignore[no-any-return]
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/forrestlin/Library/Python/3.11/lib/python/site-packages/dash/_get_app.py", line 17, in wrap
    return ctx.run(func, self, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/forrestlin/Library/Python/3.11/lib/python/site-packages/dash/dash.py", line 1600, in dispatch
    response_data = ctx.run(partial_func)
                    ^^^^^^^^^^^^^^^^^^^